# Agentic Pattern Discovery — Analysis
Load `patterns.tsv` and visualize discovery progress.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load data
df = pd.read_csv('patterns.tsv', sep='\t')
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Total experiments: {len(df)}")
print(f"Keep: {len(df[df.status == 'keep'])}")
print(f"Refine: {len(df[df.status == 'refine'])}")
print(f"Discard: {len(df[df.status == 'discard'])}")

In [ ]:
# Composite score over time
fig, ax = plt.subplots(figsize=(14, 6))

colors = {'keep': '#2ecc71', 'refine': '#f39c12', 'discard': '#95a5a6'}
for status, group in df.groupby('status'):
    ax.scatter(group.index, group.composite, c=colors[status],
               label=status, s=60, alpha=0.7, zorder=3)

# Annotate keep patterns
for _, row in df[df.status == 'keep'].iterrows():
    ax.annotate(row.pattern_name, (row.name, row.composite),
                textcoords="offset points", xytext=(5, 5),
                fontsize=7, rotation=30)

ax.set_xlabel('Experiment #')
ax.set_ylabel('Composite Score')
ax.set_title('Pattern Discovery Progress')
ax.axhline(y=7.0, color='#2ecc71', linestyle='--', alpha=0.5, label='Keep threshold')
ax.axhline(y=6.0, color='#f39c12', linestyle='--', alpha=0.5, label='Refine threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()

In [ ]:
# Score distribution by dimension
dimensions = ['novelty', 'feasibility', 'business_value', 'simplicity',
              'vertical_depth', 'content_value', 'buildability']

fig, axes = plt.subplots(1, 7, figsize=(20, 4), sharey=True)
for ax, dim in zip(axes, dimensions):
    ax.hist(df[dim], bins=range(1, 12), color='#3498db', alpha=0.7, edgecolor='white')
    ax.set_title(dim.replace('_', '\n'), fontsize=9)
    ax.set_xlim(0.5, 10.5)

fig.suptitle('Score Distribution by Dimension', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Vertical coverage
vertical_counts = df.groupby(['vertical', 'status']).size().unstack(fill_value=0)
vertical_counts.plot(kind='barh', stacked=True, color=['#95a5a6', '#f39c12', '#2ecc71'],
                     figsize=(10, 6))
plt.title('Experiments by Vertical')
plt.xlabel('Count')
plt.legend(title='Status')
plt.tight_layout()
plt.show()

In [ ]:
# Top patterns leaderboard
keeps = df[df.status == 'keep'].sort_values('composite', ascending=False)
if len(keeps) > 0:
    print("Top Patterns:")
    for _, row in keeps.head(10).iterrows():
        print(f"  {row.composite:.2f}  {row.pattern_name} ({row.vertical}) — {row.one_liner}")
else:
    print("No keep patterns yet.")